In [1]:
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
import numpy as np
import pyreadr
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [2]:
resultado = pyreadr.read_r(r"..\dados\comScrubbing\zv_pm_cc400_tc100_SITEdxGroup_spCorr_tcOk.RData")
phenotipo = pyreadr.read_r(r"..\dados\phenotypeComplete.RData")

rois = pyreadr.read_r(r"..\dados\roisInteresse.RData")["roisInteresse"]
rois.columns = ["Pertence"]
rois["Rois"] = [f"ROI_{i}" for i in range(1, 352)]
rois_interesse = rois[rois["Pertence"] == True].reset_index(drop=True)

cluster_rois = pyreadr.read_r(r"..\dados\labelsComScrubbing_Nclusters_collection\labelsComScrubbing_5clusters.RData")

In [3]:
matriz_conectividade = resultado["dataset"].copy() # Matrix de connectividade de Z scores ja sem SITE effect.
idades = resultado["funcIdade"].copy() # Idade
sexo = resultado["funcSexo"].copy() # Sexo
grupos = resultado["dxGroup"].copy() # Grupos -  Autismo, Controle e Aspenger 
tcSize = resultado["tcSize"].copy() # Quantidade de scans viaveis depois do Scrubbing
tcSize.columns = ["tcSize"]

clusters = cluster_rois["labels"].astype("int").copy() # Label do cluster de cada ROI 
df_phenotipo = phenotipo["phenotype"].copy() # Fenotipo completo dos pacientes. (Idades sujas, cuidado!!!!)

In [4]:
rois_interesse = pd.merge(rois_interesse,
                            clusters,
                            left_index=True,
                            right_index=True
                            )

In [5]:
df_phenotipo[["FIQ", "PIQ", "VIQ"]]

,FIQ,PIQ,VIQ
0,93.0,108,80
1,106.0,118,94
2,133.0,122,135
3,107.0,110,102
4,102.0,103,101
...,...,...,...
891,72.0,67,57
892,119.0,126,101
893,76.0,93,79
894,88.0,94,86


In [6]:
display(df_phenotipo[(df_phenotipo["FIQ"].isna())|(df_phenotipo["FIQ"] < 0)]["FIQ"].value_counts(dropna=False))
display(df_phenotipo[(df_phenotipo["PIQ"].isna())|(df_phenotipo["PIQ"] < 0)]["PIQ"].value_counts(dropna=False))
display(df_phenotipo[(df_phenotipo["VIQ"].isna())|(df_phenotipo["VIQ"] < 0)]["VIQ"].value_counts(dropna=False))

FIQ
 NaN       35
-9999.0    28
Name: count, dtype: int64

PIQ
NaN      89
-9999    20
Name: count, dtype: int64

VIQ
NaN      106
-9999     16
Name: count, dtype: int64

### Construção dos raios espectrais

In [7]:
index_controle = grupos[grupos["dxGroup"] == 1].index
index_autismo = grupos[grupos["dxGroup"] == 2].index

matriz_z_score_controle = matriz_conectividade[index_controle, :, :]
matriz_z_score_autismo = matriz_conectividade[index_autismo, :, :]

idade_sexo = pd.concat([idades, sexo], axis=1)

In [8]:
print(matriz_z_score_controle.shape)
print(matriz_z_score_autismo.shape)

(529, 316, 316)
(285, 316, 316)


In [11]:
def z_score_to_p_valores(z_scores):
    z_absoluto = np.abs(z_scores)
    p_valores = 2 * norm.sf(z_absoluto, loc=0, scale=1)
    return p_valores


def constroi_matrizes_adj(z_scores, alpha=0.05, ajuste="fdr_bh"):
    matriz_p_valor = z_score_to_p_valores(z_scores)
    matriz_completa = []

    for _, matriz_p in enumerate(matriz_p_valor):
        indices_triang_sup = np.triu_indices_from(matriz_p, k=1) # Sem diagonal
        p_valores_unicos = matriz_p[indices_triang_sup]

        # 2. Aplicar a correção FDR
        _, p_corrigidos_fdr, _, _ = multipletests(p_valores_unicos, alpha=alpha, method=ajuste)

        # 3. Reconstruir a matriz com os p-valores corrigidos
        matriz_p_corrigida = np.zeros_like(matriz_p)
        matriz_p_corrigida[indices_triang_sup] = p_corrigidos_fdr
        matriz_p_corrigida = matriz_p_corrigida + matriz_p_corrigida.T
        np.fill_diagonal(matriz_p_corrigida, 1)

        matriz_completa.append(matriz_p_corrigida)

    matriz_completa = np.array(matriz_completa)
    matriz_adj = (matriz_completa < alpha).astype(int)
    return matriz_adj


def constroi_raios_espectrais(matriz_adj, clusters):
    dict = {}
    clusters_unicos = clusters["labels"].unique()
    for cluste in clusters_unicos:
        rois  =clusters[clusters['labels'] == cluste]["Rois"].values
        print(f"Numero de vertices: {len(rois)}")
        print(f"Cluster {cluste}: {rois}\n")
        index_rois = clusters[clusters["labels"] == cluste].index
        idx_matrizes = np.arange(matriz_adj.shape[0])
        tensor = matriz_adj[np.ix_(idx_matrizes, index_rois, index_rois)]
        dict[f"Cluster_{cluste}"] = []

        for matriz in tensor:
            autovalores = np.linalg.eigvalsh(matriz)
            maior_autovalor = autovalores[-1] 
            dict[f"Cluster_{cluste}"].append(maior_autovalor)

    df_autovalores = pd.DataFrame(dict)
    return df_autovalores

In [12]:
matriz_adj_controle = constroi_matrizes_adj(matriz_z_score_controle)
matriz_adj_autismo = constroi_matrizes_adj(matriz_z_score_autismo)

print(f"################ CONTROLE #################")
df_controle = constroi_raios_espectrais(matriz_adj_controle, rois_interesse)

print(f"################ AUTISMO #################") 
df_autismo = constroi_raios_espectrais(matriz_adj_autismo, rois_interesse)

################ CONTROLE #################
Numero de vertices: 69
Cluster 1: ['ROI_1' 'ROI_21' 'ROI_22' 'ROI_25' 'ROI_29' 'ROI_45' 'ROI_52' 'ROI_54'
 'ROI_56' 'ROI_57' 'ROI_58' 'ROI_59' 'ROI_66' 'ROI_76' 'ROI_80' 'ROI_91'
 'ROI_92' 'ROI_97' 'ROI_110' 'ROI_111' 'ROI_116' 'ROI_122' 'ROI_124'
 'ROI_129' 'ROI_131' 'ROI_135' 'ROI_141' 'ROI_147' 'ROI_149' 'ROI_152'
 'ROI_155' 'ROI_161' 'ROI_170' 'ROI_177' 'ROI_178' 'ROI_179' 'ROI_180'
 'ROI_186' 'ROI_190' 'ROI_194' 'ROI_202' 'ROI_203' 'ROI_204' 'ROI_205'
 'ROI_207' 'ROI_213' 'ROI_223' 'ROI_228' 'ROI_231' 'ROI_236' 'ROI_237'
 'ROI_245' 'ROI_251' 'ROI_255' 'ROI_262' 'ROI_267' 'ROI_269' 'ROI_271'
 'ROI_277' 'ROI_290' 'ROI_295' 'ROI_299' 'ROI_319' 'ROI_328' 'ROI_336'
 'ROI_338' 'ROI_344' 'ROI_345' 'ROI_346']

Numero de vertices: 56
Cluster 2: ['ROI_2' 'ROI_19' 'ROI_28' 'ROI_36' 'ROI_42' 'ROI_44' 'ROI_60' 'ROI_63'
 'ROI_68' 'ROI_83' 'ROI_84' 'ROI_90' 'ROI_94' 'ROI_96' 'ROI_102' 'ROI_105'
 'ROI_117' 'ROI_134' 'ROI_136' 'ROI_142' 'ROI_146' 'ROI_14

In [29]:
mapping_columns = {"Cluster_1": "Somatomotor",
                   "Cluster_2": "Visual",
                   "Cluster_3": "Default_mode",
                   "Cluster_4": "Cerebellar",
                   "Cluster_5": "Control"
                   }

df_controle.rename(columns=mapping_columns, inplace=True)
df_autismo.rename(columns=mapping_columns, inplace=True)

- Somatomotor: 69 vertices (ROIS)
- Visual: 56
- Default Mode: 92
- Cerebellar: 45
- Control: 54

In [30]:
df_controle["idade"] = idades.loc[index_controle].reset_index(drop=True)
df_autismo["idade"] = idades.loc[index_autismo].reset_index(drop=True)

df_controle["sexo"] = sexo.loc[index_controle].reset_index(drop=True)
df_autismo["sexo"] = sexo.loc[index_autismo].reset_index(drop=True)

- Cluster 1: Somatomotor
- Cluster 2: Visual
- Cluster 3: Default-mode
- Cluster 4: Cerebellar
- Cluster 5: Control (Cognitive Control)

In [31]:
#df_autismo.to_csv(r"..\dados\raios_espectrais\raios_espectrais_autismo.csv", index=False)
#df_controle.to_csv(r"..\dados\raios_espectrais\raios_espectrais_controle.csv", index=False)

Retirar efeito dentro dos raios espectrais

In [32]:

def remover_efeito_idade_sexo(df, variavel_alvo, col_idade='idade', col_sexo='sexo', family=sm.families.Gaussian()):
    """
    Remove o efeito de idade e sexo de uma variável usando GLM.
    
    Parâmetros:
    - df: DataFrame do Pandas contendo os dados.
    - variavel_alvo: String com o nome da coluna que você quer "limpar".
    - col_idade: String com o nome da coluna de idade.
    - col_sexo: String com o nome da coluna de sexo.
    - family: Família de distribuição do GLM (padrão é Gaussiana para variáveis contínuas).
    
    Retorna:
    - Uma Série do Pandas com os valores ajustados.
    - O modelo ajustado do statsmodels (caso queira inspecionar p-valores, etc).
    """
    
    # 1. Monta a fórmula do modelo. 
    # O "C()" garante que a variável sexo seja tratada como categórica (útil se estiver como 0/1)
    formula = f"{variavel_alvo} ~ {col_idade} + C({col_sexo})"
    
    # 2. Ajusta o modelo GLM
    modelo = smf.glm(formula=formula, data=df, family=family).fit()
    
    # 3. Extrai os resíduos (Y_real - Y_previsto)
    # resid_response é o resíduo puro na mesma escala da variável resposta
    residuos = modelo.resid_response 
    
    # 4. Soma a média original para manter a variável na mesma escala (opcional, mas recomendado)
    media_original = df[variavel_alvo].mean()
    variavel_ajustada = residuos + media_original
    
    return variavel_ajustada, modelo

In [33]:
for c in mapping_columns.values():
    print(c)
    df_controle[c], _ = remover_efeito_idade_sexo(df_controle, c)
    df_autismo[c], _ = remover_efeito_idade_sexo(df_autismo, c)

colunas = ["idade", "sexo"]
df_autismo.drop(columns=colunas, inplace=True)
df_controle.drop(columns=colunas, inplace=True)

Somatomotor
Visual
Default_mode
Cerebellar
Control


In [34]:
df_autismo_prep = df_autismo.rank(pct=True)
df_controle_prep = df_controle.rank(pct=True)

In [35]:
df_autismo.to_csv(r"..\dados\raios_espectrais\raios_espectrais_SitioIdadeSexo_autismo.csv", index=False)
df_controle.to_csv(r"..\dados\raios_espectrais\raios_espectrais_SitioIdadeSexo_controle.csv", index=False)

df_autismo_prep.to_csv(r"..\dados\raios_espectrais\raios_espectrais_SitioIdadeSexo_autismo_prep.csv", index=False)
df_controle_prep.to_csv(r"..\dados\raios_espectrais\raios_espectrais_SitioIdadeSexo_controle_prep.csv", index=False)